# Quantitative Value Strategy

**Author: Abraham Sobowale**

A value-investing screen over the S&P 500. In this project I rank stocks on valuation metrics (such as the price-to-earnings ratio, and a composite of several multiples) to select the cheapest names, then size an equal-weight portfolio of them.

Built from the FreeCodeCamp *Algorithmic Trading in Python* template and reworked into my own implementation.

In [2]:
import numpy as np
import pandas as pd
import requests
import math
from scipy import stats
from scipy.stats import percentileofscore as score
import xlsxwriter
from datetime import date, timedelta

## Importing Our List of Stocks & API Token


In [3]:
api_key= "UTyHVduohyBpg6IFqApjemQi7MtN1dTB"
stocks = pd.read_csv("sp_500_stocks.csv")
today = date.today()
one_year_ago = today - timedelta(days=(365))

## Making Our First API Call


In [4]:
symbol = "GM"
api_call = "https://financialmodelingprep.com/stable/company-screener?apikey=UTyHVduohyBpg6IFqApjemQi7MtN1dTB"
data = requests.get(api_call).json()
len(data)

1000

## Parsing Our API Call


In [5]:
price = data["price"]
pe_ratio = data["pe"]
pe_ratio

TypeError: list indices must be integers or slices, not str

## Executing A Batch API Call & Building Our DataFrame


In [6]:
# Function sourced from 
# https://stackoverflow.com/questions/312443/how-do-you-split-a-list-into-evenly-sized-chunks
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]   
        
symbol_groups = list(chunks(stocks['Ticker'], 100))
symbol_strings = []
for i in range(0, len(symbol_groups)):
    symbol_strings.append(','.join(symbol_groups[i]))
#     print(symbol_strings[i])

my_columns = ['Ticker', 'Price', 'Price-to-Earnings Ratio', 'Number of Shares to Buy']

Now I need to create a blank DataFrame and add our data to the data frame one-by-one.

In [7]:
symbol_groups = list(chunks(stocks["Ticker"], 100))
symbol_str = []
for i in range(len(symbol_groups)):
    symbol_str.append(",".join(symbol_groups[i]))

final_df = pd.DataFrame(columns=my_columns)
rows = []

for symbol_s in symbol_str:
    batch_api_call_url = f"https://financialmodelingprep.com/api/v3/quote/{symbol_s}?apikey=UTyHVduohyBpg6IFqApjemQi7MtN1dTB"
    data = requests.get(batch_api_call_url).json()
    #print(data)
    for stock_data in data:  # Loop through each stock's data in the batch response
        symbol = stock_data["symbol"]  # Get the stock symbol from the response
        rows.append({
            my_columns[0]: symbol,
            my_columns[1]: stock_data["price"],
            my_columns[2]: stock_data.get("pe", "N/A"),  # Use .get() to handle missing keys
            my_columns[3]: "N/A"  # Or any default value you want
        })
    
final_df = pd.DataFrame(rows, columns=my_columns) 
final_df

,Ticker,Price,Price-to-Earnings Ratio,Number of Shares to Buy
0,A,114.8000,28.28,N/A
1,AAL,11.2150,13.35,N/A
2,AAP,53.3850,-5.49,N/A
3,AAPL,203.9900,28.10,N/A
4,ABBV,196.1750,92.97,N/A
...,...,...,...,...
489,YUM,146.6200,29.27,N/A
490,ZBH,92.2400,20.50,N/A
491,ZBRA,341.6734,32.36,N/A
492,ZION,52.8549,9.65,N/A


## Removing Glamour Stocks




In [8]:
final_df.sort_values("Price-to-Earnings Ratio", ascending = False, inplace = True)
final_df = final_df[final_df["Price-to-Earnings Ratio"] > 10]
final_df = final_df[:50]
final_df.reset_index(drop = True, inplace = True)
final_df

,Ticker,Price,Price-to-Earnings Ratio,Number of Shares to Buy
0,DD,70.6000,2353.330000,N/A
1,COF,209.9250,599.790000,N/A
2,KSU,293.5900,277.495274,N/A
3,IRM,94.9750,231.650000,N/A
4,TWTR,53.7000,214.800000,N/A
5,VTR,68.1050,158.380000,N/A
6,AMD,176.8950,129.120000,N/A
7,NOW,926.0300,116.340000,N/A
8,AVGO,297.8320,108.700000,N/A
9,QRVO,84.8750,99.850000,N/A


## Calculating the Number of Shares to Buy
We now need to calculate the number of shares I need to buy. 

To do this, I will use the `portfolio_input` function that I created in our momentum project.

I have included this function below.

In [9]:
def budget():
    portfolio_size = input("What is your portfolio size? ")
    
    while True:
        try:
            val = float(portfolio_size)
            print(portfolio_size)
            break  # Exit the loop if conversion succeeds
        except ValueError:
            print("Please enter a valid number.")
            portfolio_size = input("What is your portfolio size? ")    
    return val

Use the `portfolio_input` function to accept a `portfolio_size` variable from the user of this script.

You can now use the global `portfolio_size` variable to calculate the number of shares that our strategy should purchase.

In [10]:
position_size = budget()/len(final_df.index)
for i in range(len(final_df.index)):    
    final_df["Number of Shares to Buy"] = (position_size / final_df["Price"]).apply(math.floor)
final_df

What is your portfolio size?  45431


45431


,Ticker,Price,Price-to-Earnings Ratio,Number of Shares to Buy
0,DD,70.6000,2353.330000,12
1,COF,209.9250,599.790000,4
2,KSU,293.5900,277.495274,3
3,IRM,94.9750,231.650000,9
4,TWTR,53.7000,214.800000,16
5,VTR,68.1050,158.380000,13
6,AMD,176.8950,129.120000,5
7,NOW,926.0300,116.340000,0
8,AVGO,297.8320,108.700000,3
9,QRVO,84.8750,99.850000,10


## Building a Better (and More Realistic) Value Strategy
Every valuation metric has certain flaws.

For example, the price-to-earnings ratio doesn't work well with stocks with negative earnings.

Similarly, stocks that buyback their own shares are difficult to value using the price-to-book ratio.

Investors typically use a `composite` basket of valuation metrics to build robust quantitative value strategies. In this section, I will filter for stocks with the lowest percentiles on the following metrics:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* Enterprise Value divided by Earnings Before Interest, Taxes, Depreciation, and Amortization (EV/EBITDA)
* Enterprise Value divided by Gross Profit (EV/GP)

Some of these metrics aren't provided directly by the IEX Cloud API, and must be computed after pulling raw data. We'll start by calculating each data point from scratch.

In [11]:
symbol_groups = list(chunks(stocks["Ticker"], 100))
symbol_str = []
for i in range(len(symbol_groups)):
    symbol_str.append(",".join(symbol_groups[i]))

comp_df = pd.DataFrame(columns=my_columns)
rows = []

for symbol_s in symbol_str:
    batch_api_call_url1 = f"https://financialmodelingprep.com/api/v3/quote/{symbol_s}?apikey={api_key}"
    batch_api_call_url2 = f"https://financialmodelingprep.com/stable/key-metrics-ttm?symbol={symbol_s}&apikey={api_key}"
    batch_api_call_url3 = f"https://financialmodelingprep.com/api/v3/balance-sheet-statement/?period=annual&apikey={api_key}"
    data1 = requests.get(batch_api_call_url1).json()
    data2 = requests.get(batch_api_call_url1).json()
    #print(data)
    for stock_data1, stock_data2 in zip(data1, data2):  # Loop through each stock's data in the batch response
        symbol = stock_data1["symbol"]  # Get the stock symbol from the response
        rows.append({
            my_columns[0]: symbol,
            my_columns[1]: stock_data1["price"],
            my_columns[2]: stock_data1.get("pe", "N/A"),  # Use .get() to handle missing keys
            my_columns[3]: "N/A"  # Or any default value you want
        })
    
comp_df = pd.DataFrame(rows, columns=my_columns) 
comp_df

Let's move on to building our DataFrame. You'll notice that I use the abbreviation `rv` often. It stands for `robust value`, which is what I'll call this sophisticated strategy moving forward.

In [49]:
data1 = "hi"
data1

'hi'

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,100.010,N/A,46.95,N/A,6.596140,N/A,26.372976,N/A,11.311629,N/A,N/A
1,AAL,13.360,N/A,-1.71,N/A,-60.417952,N/A,5.952664,N/A,3.098483,N/A,N/A
2,AAP,161.440,N/A,29,N/A,3.123759,N/A,15.086368,N/A,3.208667,N/A,N/A
3,AAPL,466.070,N/A,34.69,N/A,22.373999,N/A,25.708773,N/A,19.428993,N/A,N/A
4,ABBV,96.680,N/A,21,N/A,-21.463532,N/A,12.272585,N/A,7.672427,N/A,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,94.320,N/A,28,N/A,-3.659682,N/A,18.841249,N/A,13.891510,N/A,N/A
501,ZBH,143.470,N/A,718.1,N/A,2.390128,N/A,17.170711,N/A,7.478460,N/A,N/A
502,ZBRA,288.222,N/A,31.86,N/A,8.600669,N/A,19.480804,N/A,8.449885,N/A,N/A
503,ZION,35.770,N/A,13.24,N/A,0.766237,N/A,NaN,N/A,NaN,N/A,N/A


{'week52change': 0.290414, 'week52high': 169.51, 'week52low': 90.41, 'marketcap': 75861250162, 'employees': 10724, 'day200MovingAvg': 134.17, 'day50MovingAvg': 148.64, 'float': 479955020, 'avg10Volume': 1641586.3, 'avg30Volume': 1479579.2, 'ttmEPS': 3.499, 'ttmDividendRate': 0.77, 'companyName': 'Zoetis, Inc.', 'sharesOutstanding': 494883276, 'maxChangePercent': 3.3525, 'year5ChangePercent': 1.69, 'year2ChangePercent': 0.608, 'year1ChangePercent': 0.281583, 'ytdChangePercent': 0.189876, 'month6ChangePercent': 0.10241, 'month3ChangePercent': 0.297048, 'month1ChangePercent': 0.159224, 'day30ChangePercent': 0.165634, 'day5ChangePercent': -0.012817, 'nextDividendDate': None, 'dividendYield': 0.00489706569361443, 'nextEarningsDate': '2020-10-13', 'exDividendDate': '2020-07-01', 'peRatio': 49.1, 'beta': 0.9584403132945124, 'totalCash': 1974321215, 'currentDebt': 549171552, 'revenue': 6518431250, 'grossProfit': 4133660573, 'totalRevenue': 6300683466, 'EBITDA': 2502772553, 'revenuePerShare': 1

## Dealing With Missing Data in Our DataFrame

Our DataFrame contains some missing data because all of the metrics I require are not available through the API I'm using. 

You can use pandas' `isnull` method to identify missing data:

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
17,AFL,39.060,N/A,9.7,N/A,0.950934,N/A,NaN,N/A,NaN,N/A,N/A
18,AIG,31.950,N/A,-5.88,N/A,0.407454,N/A,NaN,N/A,NaN,N/A,N/A
20,AIZ,127.680,N/A,20.22,N/A,1.360264,N/A,NaN,N/A,NaN,N/A,N/A
26,ALL,97.880,N/A,7.1,N/A,1.173772,N/A,NaN,N/A,NaN,N/A,N/A
39,ANTM,293.560,N/A,12.24,N/A,2.304037,N/A,NaN,N/A,NaN,N/A,N/A
40,AON,196.350,N/A,26.09,N/A,13.486066,N/A,17.533688,N/A,NaN,N/A,N/A
56,BAC,26.970,N/A,12.63,N/A,0.867662,N/A,NaN,N/A,NaN,N/A,N/A
64,BK,38.160,N/A,8.27,N/A,0.804550,N/A,NaN,N/A,NaN,N/A,N/A
65,BKNG,1866.170,N/A,30.78,N/A,12.571484,N/A,12.859665,N/A,NaN,N/A,N/A
75,C,55.140,N/A,9.52,N/A,0.576968,N/A,NaN,N/A,NaN,N/A,N/A


Dealing with missing data is an important topic in data science.

There are two main approaches:

* Drop missing data from the data set (pandas' `dropna` method is useful here)
* Replace missing data with a new value (pandas' `fillna` method is useful here)

In this tutorial, I will replace missing data with the average non-`NaN` data point from that column. 

Here is the code to do this:

Now, if I run the statement from earlier to print rows that contain missing data, nothing should be returned:

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score


## Calculating Value Percentiles

We now need to calculate value score percentiles for every stock in the universe. More specifically, I need to calculate percentile scores for the following metrics for every stock:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* EV/EBITDA
* EV/GP

Here's how I'll do this:

0      0.841584
1      0.112871
2      0.623762
3      0.740594
4      0.427723
         ...   
500         0.6
501    0.994059
502    0.693069
503    0.257426
504    0.843564
Name: PE Percentile, Length: 505, dtype: object
0       0.752475
1      0.0158416
2       0.510891
3       0.940594
4      0.0257426
         ...    
500     0.049505
501     0.415842
502     0.811881
503     0.132673
504     0.956436
Name: PB Percentile, Length: 505, dtype: object
0       0.877228
1      0.0732673
2        0.50099
3       0.861386
4       0.350495
         ...    
500     0.744554
501     0.572277
502     0.762376
503      0.69505
504     0.924752
Name: EV/EBITDA Percentile, Length: 505, dtype: object
0       0.552475
1      0.0574257
2      0.0653465
3       0.865347
4       0.340594
         ...    
500     0.744554
501     0.326733
502          0.4
503     0.644554
504     0.869307
Name: EV/GP Percentile, Length: 505, dtype: object


,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,100.010,N/A,46.95,0.841584,6.596140,0.752475,26.372976,0.877228,11.311629,0.552475,N/A
1,AAL,13.360,N/A,-1.71,0.112871,-60.417952,0.0158416,5.952664,0.0732673,3.098483,0.0574257,N/A
2,AAP,161.440,N/A,29.00,0.623762,3.123759,0.510891,15.086368,0.50099,3.208667,0.0653465,N/A
3,AAPL,466.070,N/A,34.69,0.740594,22.373999,0.940594,25.708773,0.861386,19.428993,0.865347,N/A
4,ABBV,96.680,N/A,21.00,0.427723,-21.463532,0.0257426,12.272585,0.350495,7.672427,0.340594,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,94.320,N/A,28.00,0.6,-3.659682,0.049505,18.841249,0.744554,13.891510,0.744554,N/A
501,ZBH,143.470,N/A,718.10,0.994059,2.390128,0.415842,17.170711,0.572277,7.478460,0.326733,N/A
502,ZBRA,288.222,N/A,31.86,0.693069,8.600669,0.811881,19.480804,0.762376,8.449885,0.4,N/A
503,ZION,35.770,N/A,13.24,0.257426,0.766237,0.132673,18.729176,0.69505,12.206556,0.644554,N/A


## Calculating the RV Score
We'll now calculate our RV Score (which stands for Robust Value), which is the value score that I'll use to filter for stocks in this investing strategy.

The RV Score will be the arithmetic mean of the 4 percentile scores that I calculated in the last section.

To calculate arithmetic mean, I will use the mean function from Python's built-in statistics module.

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,100.010,N/A,46.95,0.841584,6.596140,0.752475,26.372976,0.877228,11.311629,0.552475,0.755941
1,AAL,13.360,N/A,-1.71,0.112871,-60.417952,0.0158416,5.952664,0.0732673,3.098483,0.0574257,0.0648515
2,AAP,161.440,N/A,29.00,0.623762,3.123759,0.510891,15.086368,0.50099,3.208667,0.0653465,0.425248
3,AAPL,466.070,N/A,34.69,0.740594,22.373999,0.940594,25.708773,0.861386,19.428993,0.865347,0.85198
4,ABBV,96.680,N/A,21.00,0.427723,-21.463532,0.0257426,12.272585,0.350495,7.672427,0.340594,0.286139
...,...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,94.320,N/A,28.00,0.6,-3.659682,0.049505,18.841249,0.744554,13.891510,0.744554,0.534653
501,ZBH,143.470,N/A,718.10,0.994059,2.390128,0.415842,17.170711,0.572277,7.478460,0.326733,0.577228
502,ZBRA,288.222,N/A,31.86,0.693069,8.600669,0.811881,19.480804,0.762376,8.449885,0.4,0.666832
503,ZION,35.770,N/A,13.24,0.257426,0.766237,0.132673,18.729176,0.69505,12.206556,0.644554,0.432426


## Selecting the 50 Best Value Stocks¶

As before, I can identify the 50 best value stocks in our universe by sorting the DataFrame on the RV Score column and dropping all but the top 50 entries.

## Calculating the Number of Shares to Buy
We'll use the `portfolio_input` function that I created earlier to accept our portfolio size. Then I will use similar logic in a for loop to calculate the number of shares to buy for each stock in our investment universe.

Enter the value of your portfolio:1000000


,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,HPE,9.830,1994,-293.68,0.0118812,0.768699,0.134653,4.757379,0.0455446,2.575722,0.039604,0.0579208
1,FTI,8.870,2210,-0.68,0.134653,0.526579,0.0732673,2.722313,0.00792079,1.745265,0.0178218,0.0584158
2,AAL,13.360,1467,-1.71,0.112871,-60.417952,0.0158416,5.952664,0.0732673,3.098483,0.0574257,0.0648515
3,CCL,15.610,1256,-3.90,0.10297,0.380999,0.0554455,3.751649,0.0237624,3.458441,0.0811881,0.0658416
4,HFC,25.660,764,-26.18,0.0435644,0.716944,0.120792,3.301222,0.0118812,3.693071,0.0891089,0.0663366
5,HPQ,19.140,1024,9.17,0.172277,-23.323972,0.0237624,5.822424,0.0673267,2.516859,0.0356436,0.0747525
6,XRX,17.800,1101,9.35,0.174257,0.640297,0.0990099,3.620185,0.019802,1.604068,0.0138614,0.0767327
7,TPR,16.010,1224,-20.58,0.049505,1.200922,0.231683,3.870569,0.029703,1.149885,0.0019802,0.0782178
8,LB,27.330,717,-10.49,0.0633663,-5.047485,0.0455446,8.144139,0.170297,3.119623,0.0613861,0.0851485
9,SYF,25.300,775,7.77,0.152475,0.971730,0.180198,2.129599,0.0039604,1.604512,0.0158416,0.0881188
